In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data_prep import generate_synthetic_data, clean_data, create_and_populate_db
from model import load_modelling_data, engineer_features, preprocess_and_split, train_models, evaluate_model, run_modelling_pipeline

print('all libraries and custom modules imported successfully.')


## 1. Data Generation & Database Population

In [ ]:
raw_df = generate_synthetic_data(1000)
clean_df = clean_data(raw_df)
create_and_populate_db(clean_df, db_path='loan_risk.db')
clean_df.head()


## 2. SQL Credit Risk Analytics

In [ ]:
conn = sqlite3.connect('loan_risk.db')

# query 1: overall default rate
q1 = '''
SELECT 
    COUNT(*) AS total_loans,
    SUM(default_flag) AS total_defaulters,
    ROUND(AVG(default_flag) * 100.0, 2) AS default_rate_pct
FROM repayment_status;
'''
print('overall portfolio default rate:')
display(pd.read_sql_query(q1, conn))


In [ ]:
# query 2: default rate by loan purpose
q2 = '''
SELECT 
    l.loan_purpose,
    COUNT(l.loan_id) AS total_loans,
    SUM(r.default_flag) AS defaulter_count,
    ROUND(AVG(r.default_flag) * 100.0, 2) AS default_rate_pct
FROM loans l
JOIN repayment_status r ON l.loan_id = r.loan_id
GROUP BY l.loan_purpose
ORDER BY default_rate_pct DESC;
'''
print('default rate grouped by loan purpose:')
display(pd.read_sql_query(q2, conn))


In [ ]:
# query 9: cte ranking loan purposes by default rate
q9 = '''
WITH purpose_risk AS (
    SELECT 
        l.loan_purpose,
        COUNT(l.loan_id) AS total_loans,
        SUM(r.default_flag) AS total_defaults,
        ROUND(AVG(r.default_flag) * 100.0, 2) AS default_rate_pct
    FROM loans l
    JOIN repayment_status r ON l.loan_id = r.loan_id
    GROUP BY l.loan_purpose
)
SELECT 
    loan_purpose,
    total_loans,
    total_defaults,
    default_rate_pct,
    DENSE_RANK() OVER (ORDER BY default_rate_pct DESC) AS risk_rank
FROM purpose_risk;
'''
print('cte risk ranking across loan purposes:')
display(pd.read_sql_query(q9, conn))
conn.close()


## 3. Machine Learning Modeling Pipeline

In [ ]:
run_modelling_pipeline()

## 4. Visualizations & Risk Insights

In [ ]:
from IPython.display import Image, display

display(Image(filename='chart_default_rate_by_segment.png'))
display(Image(filename='chart_confusion_matrices.png'))
display(Image(filename='chart_roc_curves.png'))
display(Image(filename='chart_feature_importances.png'))
